# Equity Market Analytics & Portfolio Risk Analysis
### Quantitative Finance Research & Portfolio Optimization Workflow
This notebook demonstrates the end-to-end analytical pipeline: data ingestion into SQLite, return analytics, risk metrics calculation, portfolio risk attribution, and Markowitz Efficient Frontier optimization.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.database import DatabaseManager
from src.data_loader import MarketDataLoader
from src.returns import calculate_daily_returns, calculate_cumulative_returns, calculate_annualized_return
from src.risk_metrics import calculate_annualized_volatility, calculate_sharpe_ratio, calculate_sortino_ratio, calculate_max_drawdown, calculate_historical_var, calculate_beta
from src.portfolio import calculate_portfolio_daily_returns, calculate_portfolio_metrics, calculate_risk_contribution, generate_efficient_frontier
from src.analytics import generate_asset_summary_table
from src.visualization import plot_normalized_prices, plot_correlation_heatmap, plot_efficient_frontier, plot_portfolio_allocation

%matplotlib inline

## 1. Ingest Market Data & Initialize SQLite Database

In [ ]:
db = DatabaseManager("../data/market_data.db")
loader = MarketDataLoader(db)
tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "TSLA", "JPM", "META"]
benchmark = "^GSPC"

# Ingest 3 years of daily OHLCV data
prices_df = loader.get_adjusted_close_matrix(tickers, start_date="2021-01-01")
bench_df = loader.get_adjusted_close_matrix([benchmark], start_date="2021-01-01")
bench_ret = calculate_daily_returns(bench_df[benchmark])
returns_df = calculate_daily_returns(prices_df)

print(f"Successfully ingested {len(prices_df)} price records for {len(tickers)} equities.")
prices_df.tail()

## 2. Multi-Asset Summary Tear Sheet

In [ ]:
summary_table = generate_asset_summary_table(prices_df, benchmark_returns=bench_ret, risk_free_rate=0.04)
summary_table

## 3. Correlation & Covariance Matrix

In [ ]:
fig_corr = plot_correlation_heatmap(returns_df)
plt.show()

## 4. Custom Portfolio Construction & Risk Decomposition

In [ ]:
custom_weights = {
    "AAPL": 0.25,
    "MSFT": 0.20,
    "NVDA": 0.20,
    "AMZN": 0.15,
    "GOOGL": 0.10,
    "JPM": 0.10
}

port_metrics = calculate_portfolio_metrics(returns_df, custom_weights, benchmark_returns=bench_ret, risk_free_rate=0.04)
for k, v in port_metrics.items():
    if k != "Weights":
        print(f"{k:30s}: {v:.4f}")

# Risk attribution (PCTR %)
risk_decomp = calculate_risk_contribution(returns_df, custom_weights)
risk_decomp

## 5. Markowitz Mean-Variance Efficient Frontier

In [ ]:
frontier = generate_efficient_frontier(returns_df, risk_free_rate=0.04, num_portfolios=3000)
fig_ef = plot_efficient_frontier(frontier, risk_free_rate=0.04)
plt.show()

print("Max Sharpe Allocation:", frontier["max_sharpe"]["weights"])
print("Min Volatility Allocation:", frontier["min_volatility"]["weights"])